# Week 10 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-10.md`](../../weeks/week-10.md) · **Given engine:** [`localsearch.py`](localsearch.py)

Teaching walkthrough of the week-10 studio (genetic algorithms & local search on TSP).
It **imports the reference code from [`solution.py`](solution.py)** — never re-pastes it
— so the answer shown here is byte-for-byte the one the provided test grades.
Correctness is proved separately: `python3 studios/_verify_solutions.py week-10` runs
the unmodified `test_localsearch.py` against `solution.py`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: put the week folder (for solution/localsearch) and the repo root
# (for aicourse) on the path, so this runs from anywhere. ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-10"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect, random
import numpy as np
import localsearch
from localsearch import load_tsp, validate_tour, one_point_crossover, tsp_hill_climb, tsp_genetic_algorithm
import solution

TSP = load_tsp()
N = TSP.n
print(f"instance: {N} cities")

## The trap — plain one-point crossover breaks the permutation

Before OX: splicing the head of one tour onto the tail of another produces
repeated/missing cities. The GIVEN `one_point_crossover` is deliberately broken so
`validate_tour` catches it immediately. Mirrors `test_one_point_crossover_is_invalid`.

In [ ]:
invalid = 0
for i in range(200):
    rng = random.Random(1000 + i)
    p1, p2 = TSP.random_tour(rng), TSP.random_tour(rng)
    if not validate_tour(one_point_crossover(p1, p2, rng), N):
        invalid += 1
assert invalid > 0
print(f"one_point_crossover produced an INVALID tour on {invalid}/200 parent pairs")
print("-> this is why permutation problems need order crossover (OX)")

## Task 1 — the two genetic operators

`order_crossover` (OX): copy a segment from parent1, fill the rest from parent2 in
order, skipping cities already taken — the segment-copy + ordered-fill is what
guarantees a valid permutation. `swap_mutation`: exchange two positions (a swap can
never duplicate). From `solution.py`:

In [ ]:
print(inspect.getsource(solution.order_crossover))
print(inspect.getsource(solution.swap_mutation))

In [ ]:
# Mirror test_order_crossover_produces_valid_tours + test_swap_mutation_valid_permutation.
for i in range(300):
    rng = random.Random(i)
    p1, p2 = TSP.random_tour(rng), TSP.random_tour(rng)
    assert validate_tour(solution.order_crossover(p1, p2, rng), N)
for i in range(200):
    rng = random.Random(i)
    assert validate_tour(solution.swap_mutation(TSP.random_tour(rng), rng), N)
print("order_crossover valid on 300 pairs; swap_mutation valid on 200 trials")

# Sanity: with working operators the GA crushes random search.
rng = random.Random(42)
random_mean = np.mean([TSP.length(TSP.random_tour(rng)) for _ in range(100)])
ga_len, _ = tsp_genetic_algorithm(TSP, solution.order_crossover, solution.swap_mutation,
                                  random.Random(0), max_evals=20_000)
assert ga_len < 0.6 * random_mean
print(f"GA {ga_len:.1f} beats random baseline {random_mean:.1f}")

## The honest result, watched live — 2-opt beats the naive GA at equal budget

This is the payload of week 10, and the test that every year gets reported as a bug.
At an **equal evaluation budget** on this 30-city instance, a 2-opt hill climber wins
on average against a naive GA. A GA is a design space, not a magic wand — being
"biologically inspired" buys nothing for free. Mirrors
`test_honest_2opt_not_beaten_by_naive_ga`.

In [ ]:
ga_lens, hc_lens = [], []
for seed in range(6):
    ga, _ = tsp_genetic_algorithm(TSP, solution.order_crossover, solution.swap_mutation,
                                  random.Random(seed), max_evals=20_000)
    _, hc = tsp_hill_climb(TSP, random.Random(seed), max_evals=20_000)
    ga_lens.append(ga); hc_lens.append(hc)
ga_mean, hc_mean = np.mean(ga_lens), np.mean(hc_lens)
wins = sum(h <= g for h, g in zip(hc_lens, ga_lens))
assert hc_mean < ga_mean
print(f"2-opt hill climb mean = {hc_mean:.1f}   naive GA mean = {ga_mean:.1f}")
print(f"HC wins {wins}/6 seeds at equal budget")
print("\nHONEST RESULT confirmed: the tidy 'evolution wins' story does NOT hold here.")
print("Report what you measure -- comparing at equal EVALUATIONS (not generations) is the rule that makes it fair.")

## Task 2 — the hyperparameter study

`run_study` sweeps population x mutation rate x tournament, N seeds each, at a fixed
**evaluation** budget (not generations — that would silently favour large populations).
From `solution.py`:

In [ ]:
print(inspect.getsource(solution.run_study))

In [ ]:
# Full study is 18 configs x 5 seeds at 20k evals; here we run a SMALL, fast slice
# (2 seeds, 3k evals) just to show the shape. Use the defaults for the real table.
rows = solution.run_study(TSP, n_seeds=2, max_evals=3_000)
print(f"{len(rows)} configs (pop, mut, tourn, mean_len, std_len):")
for pop, mut, tourn, mean, std in rows[:6]:
    print(f"  pop={pop:>3} mut={mut:<5} tourn={tourn}  ->  {mean:6.1f} +/- {std:.1f}")
print("  ... (full grid + heatmap + convergence plot + crossover ablation are the deliverables)")

### No LLM this week, by design

Week 10 has no model comparison — week 11 runs the budget-normalized bake-off, and
today's methodology (error bars, equal evaluation budget) is exactly what makes that
comparison honest. Two configs whose error bars overlap are not distinguishable by this
experiment — saying so is a stronger result than crowning a winner.